In [2]:
import time
import random
import numpy as np
import torch
from torch import nn
import torch.distributions as dist
from torch.utils.data import Dataset, DataLoader
import gymnasium as gym
from tqdm import tqdm
import ale_py

from rollout_buffer import RolloutBuffer
from python_dataset import PythonListDataset
from learner import PPOLearning
from model import PPOActorCritic

In [3]:
gym.register_envs(ale_py)

env = gym.make("ALE/Pong-v5", render_mode="rgb_array")
obs, info = env.reset()

A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]


In [ ]:
def set_seed(seed=0, env=None):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if env is not None:
        try:
            env.reset(seed=seed)
        except Exception:
            pass

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    env = gym.make("ALE/Pong-v5")

    set_seed(0, env)

    obs_dim = env.observation_space.shape[0]
    act_dim = env.action_space.n

    actorcritic = PPOActorCritic(6).to(device)

    actorcritic.load_state_dict(torch.load("pretrained_models/model250.pth", map_location=device))
    actorcritic.to(device)

    trainer = PPOLearning(actorcritic, env, device)

    trainer.train_actor_critic(
        steps = 3000,
        lr = 3e-4,
        n_epochs = 4,
        n_rollout = 2000,
        gamma = 0.99,
        delta = 0.95,
        clip_eps = 0.2,
        batch_size = 256
    )